In [0]:
!pip install -r ../requirements.txt
%restart_python

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient

# Check if index is ready first
w = WorkspaceClient()
index_name = "clinton_emails.silver.email_categories_index"

index_info = w.vector_search_indexes.get_index(index_name=index_name)

if not index_info.status.ready:
    print("⏳ Index is not ready yet!")
    print(f"Status: {index_info.status.message}")
    print("\nPlease wait a few minutes and rerun this cell.")
    print("Run Cell 3 (Check index status) to monitor progress.")
else:
    print("✅ Index is ready! Performing similarity search...\n")
    
    # Use VectorSearchClient for querying
    vsc = VectorSearchClient(disable_notice=True)
    
    # For demonstration, let's use the embedding from one of the categories
    # In practice, you'd generate a query embedding for your search term
    df = spark.table("clinton_emails.silver.email_categories").limit(1)
    query_embedding = df.select("embedding").first()["embedding"]
    
    print(f"Searching with embedding vector (dimension {len(query_embedding)})...\n")
    
    # Perform similarity search
    results = vsc.get_index(
        endpoint_name="standard_demo_api_endpoint",
        index_name=index_name
    ).similarity_search(
        query_vector=query_embedding,
        columns=["id", "category", "description"],
        num_results=5
    )
    
    print("Top 5 similar categories:")
    display(results)

In [0]:
from databricks.vector_search.client import VectorSearchClient
from typing import List, Dict, Any

# Import your custom embedding utility
from shared.llm_utils import get_embedding

def get_unique_categories(
    query_texts: List[str], 
    index_name: str = "clinton_emails.silver.email_categories_index",
    endpoint_name: str = "standard_demo_api_endpoint",
    num_results_per_query: int = 3
) -> List[Dict[str, Any]]:
    """
    Generates embeddings for an array of text queries, searches a Databricks 
    Vector Search index, and returns a deduplicated list of responses.

    Args:
        query_texts: List of text strings to search for.
        index_name: The full Unity Catalog path to the vector index.
        endpoint_name: The name of the vector search endpoint.
        num_results_per_query: How many top results to retrieve per query.

    Returns:
        A list of unique dictionary records containing the matched categories.
    """
    vsc = VectorSearchClient(disable_notice=True)
    
    try:
        index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
    except Exception as e:
        print(f"Failed to connect to index: {e}")
        return []

    unique_results = {}
    
    for text in query_texts:
        try:
            # 1. Generate the embedding vector using your custom function
            query_vector = get_embedding(text)
            
            # 2. Perform similarity search using the generated vector
            response = index.similarity_search(
                query_vector=query_vector,
                columns=["id", "category", "description"],
                num_results=num_results_per_query
            )
            
            data_array = response.get("result", {}).get("data_array", [])
            
            # 3. Parse the array and deduplicate by ID
            for row in data_array:
                row_id = row[0]
                
                if row_id not in unique_results:
                    unique_results[row_id] = {
                        "id": row[0],
                        "category": row[1],
                        "description": row[2],
                        "similarity_score": row[3] 
                    }
                    
        except Exception as e:
            print(f"Error processing query '{text}': {e}")
            continue
            
    return list(unique_results.values())

In [0]:
entries = ['women peace and security', 'resolution 1325', 'nato meeting', 'nato special representative 1325', 'us dod', 'national action plan', 'azerbaijan national action plan', 'serbia national action plan', 'burmese women engagement', 'vivian redding', 'chris stevens', '2015-05-13']

exits = get_unique_categories(entries)
display(exits)

In [0]:
from databricks.vector_search.client import VectorSearchClient
from typing import List, Dict, Any
import numpy as np

# Import your custom embedding utility
from shared.llm_utils import get_embedding

def compute_rerank_score(query_text: str, candidate_text: str) -> float:
    """
    Compute a reranking score between query and candidate using cosine similarity
    of their embeddings. This provides a more accurate relevance score.
    
    Args:
        query_text: The search query text
        candidate_text: The candidate result text to score
        
    Returns:
        A similarity score between -1 and 1 (higher is better)
    """
    try:
        query_emb = np.array(get_embedding(query_text))
        candidate_emb = np.array(get_embedding(candidate_text))
        
        # Cosine similarity
        score = np.dot(query_emb, candidate_emb) / (
            np.linalg.norm(query_emb) * np.linalg.norm(candidate_emb)
        )
        return float(score)
    except Exception as e:
        print(f"Error computing rerank score: {e}")
        return 0.0

def get_unique_categories_with_rerank(
    query_texts: List[str], 
    index_name: str = "clinton_emails.silver.email_categories_index",
    endpoint_name: str = "standard_demo_api_endpoint",
    num_results_per_query: int = 3,
    rerank: bool = True,
    rerank_pool_size: int = 10
) -> List[Dict[str, Any]]:
    """
    Generates embeddings for an array of text queries, searches a Databricks 
    Vector Search index, and returns a deduplicated list of responses with optional reranking.

    Args:
        query_texts: List of text strings to search for.
        index_name: The full Unity Catalog path to the vector index.
        endpoint_name: The name of the vector search endpoint.
        num_results_per_query: How many top results to return per query (after reranking).
        rerank: If True, retrieve more candidates and rerank them for better relevance.
        rerank_pool_size: How many candidates to retrieve for reranking (only used if rerank=True).

    Returns:
        A list of unique dictionary records containing the matched categories.
    """
    vsc = VectorSearchClient(disable_notice=True)
    
    try:
        index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
    except Exception as e:
        print(f"Failed to connect to index: {e}")
        return []

    unique_results = {}
    
    for text in query_texts:
        try:
            query_vector = get_embedding(text)
            
            # Retrieve more candidates if reranking is enabled
            num_to_retrieve = rerank_pool_size if rerank else num_results_per_query
            
            response = index.similarity_search(
                query_vector=query_vector,
                columns=["id", "category", "description"],
                num_results=num_to_retrieve
            )
            
            data_array = response.get("result", {}).get("data_array", [])
            
            candidates = []
            for row in data_array:
                candidates.append({
                    "id": row[0],
                    "category": row[1],
                    "description": row[2],
                    "initial_score": row[3]
                })
            
            # Apply reranking if enabled
            if rerank and candidates:
                for candidate in candidates:
                    # Rerank based on the combined category + description text
                    candidate_text = f"{candidate['category']}: {candidate['description']}"
                    candidate['rerank_score'] = compute_rerank_score(text, candidate_text)
                
                # Sort by rerank score (descending) and take top results
                candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
                candidates = candidates[:num_results_per_query]
            
            # Deduplicate and add to results
            for candidate in candidates:
                row_id = candidate['id']
                if row_id not in unique_results:
                    result = {
                        "id": candidate['id'],
                        "category": candidate['category'],
                        "description": candidate['description'],
                        "initial_score": candidate['initial_score']
                    }
                    if rerank:
                        result['rerank_score'] = candidate['rerank_score']
                    unique_results[row_id] = result
                    
        except Exception as e:
            print(f"Error processing query '{text}': {e}")
            continue
    
    # Sort final results by rerank_score if available, otherwise by initial_score
    results = list(unique_results.values())
    if rerank and results and 'rerank_score' in results[0]:
        results.sort(key=lambda x: x['rerank_score'], reverse=True)
    else:
        results.sort(key=lambda x: x['initial_score'], reverse=True)
        
    return results

In [0]:
exits = get_unique_categories_with_rerank(entries)
display(exits)